In [1]:
import os
import pandas as pd

filespath = r"C:\Users\nishw\AppData\Roaming\MetaQuotes\Tester\D0E8209F77C8CF37AD8BF550E51FF075\Agent-127.0.0.1-3000\MQL5\Files"

# List all files
files = os.listdir(filespath)

for f in files:
    print(f)

# Read a CSV file
csv_file = os.path.join(filespath, files[0])  # replace with a specific CSV filename if needed
df = pd.read_csv(csv_file)

print(df.head())

StrategyTest_Trades.csv
Trades.csv
   DealTicket  OrderTicket                 Time  Symbol            Type  \
0           8            8  2026.01.02 01:03:19  EURUSD  DEAL_TYPE_SELL   
1           9            9  2026.01.02 01:03:19  EURUSD  DEAL_TYPE_SELL   
2          10           10  2026.01.02 01:03:19  EURUSD  DEAL_TYPE_SELL   
3          11           11  2026.01.02 01:03:19  EURUSD  DEAL_TYPE_SELL   
4          12           12  2026.01.02 01:03:19  EURUSD  DEAL_TYPE_SELL   

   Volume    Price  Profit  Commission  Swap  
0    0.01  1.17476    -0.1         0.0   0.0  
1    0.01  1.17476    -0.1         0.0   0.0  
2    0.01  1.17476    -0.1         0.0   0.0  
3    0.01  1.17476    -0.1         0.0   0.0  
4    0.01  1.17476    -0.1         0.0   0.0  


In [ ]:
from flask import Flask
import pandas as pd
import os

app = Flask(__name__)

FILES_PATH = r"C:\Users\nishw\AppData\Roaming\MetaQuotes\Tester\D0E8209F77C8CF37AD8BF550E51FF075\Agent-127.0.0.1-3000\MQL5\Files"

@app.route("/")
def home():
    html = """
    <html>
    <head>
        <title>CSV Viewer</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 20px;
            }
            table {
                border-collapse: collapse;
                margin-bottom: 40px;
                width: 100%;
            }
            th, td {
                border: 1px solid #ccc;
                padding: 6px;
                text-align: left;
            }
            th {
                background: #f2f2f2;
            }
            h2 {
                background: #eee;
                padding: 10px;
            }
        </style>
    </head>
    <body>
        <h1>CSV Files</h1>
    """

    csv_files = sorted(
        [f for f in os.listdir(FILES_PATH) if f.lower().endswith(".csv")]
    )

    for file in csv_files:
        try:
            path = os.path.join(FILES_PATH, file)
            df = pd.read_csv(path)
            df = df.head()

            html += f"<h2>{file}</h2>"
            html += df.to_html(
                index=False,
                classes="table",
                border=0
            )

        except Exception as e:
            html += f"<h2>{file}</h2>"
            html += f"<p style='color:red'>Error: {e}</p>"

    html += "</body></html>"
    return html


if __name__ == "__main__":
    app.run(debug=False, port=5100)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5100
Press CTRL+C to quit
127.0.0.1 - - [23/Jun/2026 22:10:42] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [23/Jun/2026 22:45:00] "GET / HTTP/1.1" 200 -


In [7]:
import sys
import os
sys.path.append(os.getcwd())

import mysql.connector
from mysql.connector import Error

from sqlconnection import get_connection, close_connection

conn = get_connection()

cursor = conn.cursor()
cursor.execute("SELECT DATABASE();")
print(cursor.fetchone())

close_connection(conn)

Connected to MySQL database: marketdata
('marketdata',)
MySQL connection closed


In [3]:
!pip install mysql


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.getcwd())

from sqlconnection import get_connection, close_connection

filespath = r"C:\Users\nishw\AppData\Roaming\MetaQuotes\Tester\D0E8209F77C8CF37AD8BF550E51FF075\Agent-127.0.0.1-3000\MQL5\Files"

conn = get_connection()
cursor = conn.cursor()

# STEP 1: DROP ALL TABLES (SAFE RESET)
cursor.execute("SET FOREIGN_KEY_CHECKS = 0")

cursor.execute("SHOW TABLES")
tables = cursor.fetchall()

for t in tables:
    table_name = t[0]
    cursor.execute(f"DROP TABLE IF EXISTS `{table_name}`")

cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
conn.commit()

# STEP 2: CREATE TABLES FROM FILES + LOAD DATA
files = os.listdir(filespath)

for f in files:
    filepath = os.path.join(filespath, f)

    if not f.endswith(".csv"):
        continue

    df = pd.read_csv(filepath)

    table_name = os.path.splitext(f)[0]

    cols = df.columns
    col_defs = ", ".join([f"`{c}` TEXT" for c in cols])

    cursor.execute(f"CREATE TABLE IF NOT EXISTS `{table_name}` ({col_defs})")

    for _, row in df.iterrows():
        values = tuple(str(x) for x in row)
        placeholders = ",".join(["%s"] * len(values))

        cursor.execute(
            f"INSERT INTO `{table_name}` VALUES ({placeholders})",
            values
        )
        # Explode Comments 
        
        # make duplicate of multitimeframe

        # make column

        # populate the entries

# def commentexplode():
#     print("hi")

# #     ALTER TABLE tradejournal
# # ADD COLUMN timestamp DATETIME,
# # ADD COLUMN pips DECIMAL(10,2);
# # UPDATE tradejournal
# # SET
# #     timestamp = STR_TO_DATE(
# #         TRIM(SUBSTRING_INDEX(Comment, '|', 1)),
# #         '%Y.%m.%d %H:%i'
# #     ),
# #     pips = CAST(
# #         TRIM(SUBSTRING_INDEX(Comment, '|', -1))
# #         AS DECIMAL(10,2)
# #     );




# def build_finaltable():
#     print("hi")
# # DROP TABLE IF EXISTS multitimeframe_snapshot_features;

# # CREATE TABLE multitimeframe_snapshot_features AS
# # SELECT
# #     m.*,

# #     MAX(CASE WHEN t.Type='BUY'  AND t.pips=10 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_10,
# #     MAX(CASE WHEN t.Type='BUY'  AND t.pips=20 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_20,
# #     MAX(CASE WHEN t.Type='BUY'  AND t.pips=30 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_30,
# #     MAX(CASE WHEN t.Type='BUY'  AND t.pips=40 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_40,
# #     MAX(CASE WHEN t.Type='BUY'  AND t.pips=50 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_50,
# #     MAX(CASE WHEN t.Type='BUY'  AND t.pips=60 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_60,

# #     MAX(CASE WHEN t.Type='SELL' AND t.pips=10 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_10,
# #     MAX(CASE WHEN t.Type='SELL' AND t.pips=20 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_20,
# #     MAX(CASE WHEN t.Type='SELL' AND t.pips=30 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_30,
# #     MAX(CASE WHEN t.Type='SELL' AND t.pips=40 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_40,
# #     MAX(CASE WHEN t.Type='SELL' AND t.pips=50 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_50,
# #     MAX(CASE WHEN t.Type='SELL' AND t.pips=60 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_60

# # FROM multitimeframe_snapshot m
# # LEFT JOIN tradejournal t
# #     ON t.timestamp = FROM_UNIXTIME(CAST(m.`15m_time` AS UNSIGNED))

# # GROUP BY
# #     m.`4h_time`,
# #     m.`1h_time`,
# #     m.`15m_time`,
# #     m.`4h_open`,
# #     m.`4h_high`,
# #     m.`4h_low`,
# #     m.`4h_close`,
# #     m.`4h_dir`,
# #     m.`4h_group`,
# #     m.`4h_bias`,
# #     m.`4h_phase`,
# #     m.`4h_momentum`,
# #     m.`4h_pattern`,
# #     m.`4h_atr`,
# #     m.`4h_rsi`,
# #     m.`1h_open`,
# #     m.`1h_high`,
# #     m.`1h_low`,
# #     m.`1h_close`,
# #     m.`1h_dir`,
# #     m.`1h_group`,
# #     m.`1h_bias`,
# #     m.`1h_phase`,
# #     m.`1h_momentum`,
# #     m.`1h_pattern`,
# #     m.`1h_atr`,
# #     m.`1h_rsi`,
# #     m.`15m_open`,
# #     m.`15m_high`,
# #     m.`15m_low`,
# #     m.`15m_close`,
# #     m.`15m_dir`,
# #     m.`15m_group`,
# #     m.`15m_bias`,
# #     m.`15m_phase`,
# #     m.`15m_momentum`,
# #     m.`15m_pattern`,
# #     m.`15m_atr`,
# #     m.`15m_rsi`;

        

    conn.commit()

close_connection(conn)

Connected to MySQL database: marketdata
MySQL connection closed


In [3]:
import sys
import os
import pandas as pd

sys.path.append(os.getcwd())

from sqlconnection import get_connection, close_connection

filespath = r"C:\Users\nishw\AppData\Roaming\MetaQuotes\Tester\D0E8209F77C8CF37AD8BF550E51FF075\Agent-127.0.0.1-3000\MQL5\Files"

conn = get_connection()
cursor = conn.cursor()

# =====================================================
# STEP 1: DROP ALL TABLES
# =====================================================

cursor.execute("SET FOREIGN_KEY_CHECKS = 0")

cursor.execute("SHOW TABLES")
tables = cursor.fetchall()

for t in tables:
    cursor.execute(f"DROP TABLE IF EXISTS `{t[0]}`")

cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
conn.commit()

# =====================================================
# STEP 2: LOAD ALL CSV FILES
# =====================================================

files = os.listdir(filespath)

for f in files:

    if not f.endswith(".csv"):
        continue

    filepath = os.path.join(filespath, f)

    print(f"Loading {f}...")

    df = pd.read_csv(filepath)

    table_name = os.path.splitext(f)[0]

    cols = list(df.columns)

    col_defs = ", ".join([f"`{c}` TEXT" for c in cols])

    cursor.execute(
        f"CREATE TABLE IF NOT EXISTS `{table_name}` ({col_defs})"
    )

    placeholders = ",".join(["%s"] * len(cols))

    rows = [
        tuple("" if pd.isna(x) else str(x) for x in row)
        for row in df.values
    ]

    cursor.executemany(
        f"INSERT INTO `{table_name}` VALUES ({placeholders})",
        rows
    )

    conn.commit()

# =====================================================
# STEP 3: EXPLODE tradejournal COMMENT
# =====================================================

cursor.execute("""
ALTER TABLE tradejournal
ADD COLUMN timestamp DATETIME NULL,
ADD COLUMN pips DECIMAL(10,2) NULL
""")

cursor.execute("""
UPDATE tradejournal
SET
    timestamp = STR_TO_DATE(
        TRIM(SUBSTRING_INDEX(Comment, '|', 1)),
        '%Y.%m.%d %H:%i'
    ),
    pips = CAST(
        TRIM(SUBSTRING_INDEX(Comment, '|', -1))
        AS DECIMAL(10,2)
    )
""")

conn.commit()

# =====================================================
# STEP 4: BUILD FEATURE TABLE
# =====================================================

cursor.execute("""
DROP TABLE IF EXISTS multitimeframe_snapshot_features
""")

cursor.execute("""
CREATE TABLE multitimeframe_snapshot_features AS
SELECT
    m.*,

    MAX(CASE WHEN t.Type='BUY'  AND t.pips=10 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_10,
    MAX(CASE WHEN t.Type='BUY'  AND t.pips=20 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_20,
    MAX(CASE WHEN t.Type='BUY'  AND t.pips=30 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_30,
    MAX(CASE WHEN t.Type='BUY'  AND t.pips=40 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_40,
    MAX(CASE WHEN t.Type='BUY'  AND t.pips=50 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_50,
    MAX(CASE WHEN t.Type='BUY'  AND t.pips=60 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS buy_60,

    MAX(CASE WHEN t.Type='SELL' AND t.pips=10 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_10,
    MAX(CASE WHEN t.Type='SELL' AND t.pips=20 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_20,
    MAX(CASE WHEN t.Type='SELL' AND t.pips=30 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_30,
    MAX(CASE WHEN t.Type='SELL' AND t.pips=40 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_40,
    MAX(CASE WHEN t.Type='SELL' AND t.pips=50 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_50,
    MAX(CASE WHEN t.Type='SELL' AND t.pips=60 THEN CAST(t.Profit AS DECIMAL(10,2)) END) AS sell_60

FROM multitimeframe_snapshot m

LEFT JOIN tradejournal t
    ON t.timestamp = FROM_UNIXTIME(
        CAST(m.`15m_time` AS UNSIGNED)
    )

GROUP BY m.`15m_time`
""")

conn.commit()

print("Database rebuilt successfully.")

close_connection(conn)

Connected to MySQL database: marketdata
Loading multitimeframe_snapshot.csv...
Loading TradeJournal.csv...
Database rebuilt successfully.
MySQL connection closed


In [4]:
import pandas as pd

from sqlconnection import get_connection, close_connection

conn = get_connection()

query = """
SELECT *
FROM multitimeframe_snapshot_features
"""

df = pd.read_sql_query(query, conn)

close_connection(conn)

print(df.shape)
df.head()

Connected to MySQL database: marketdata
MySQL connection closed
(1536, 51)


C:\Users\nishw\AppData\Local\Temp\ipykernel_28444\1962548011.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,4h_time,4h_open,4h_high,4h_low,4h_close,4h_dir,4h_group,4h_bias,4h_phase,4h_momentum,...,buy_30,buy_40,buy_50,buy_60,sell_10,sell_20,sell_30,sell_40,sell_50,sell_60
0,1780084800,1.1672,1.16734,1.16584,1.16606,bearish,1,,,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1780084800,1.1672,1.16734,1.16584,1.16606,bearish,1,,,,...,-0.10,-0.10,-0.10,-0.10,-0.10,-0.10,-0.10,-0.10,-0.10,-0.10
2,1780084800,1.1672,1.16734,1.16584,1.16606,bearish,1,,,,...,0.31,0.40,-0.10,-0.10,-0.10,-0.10,-0.10,-0.10,-0.10,-0.10
3,1780084800,1.1672,1.16734,1.16584,1.16606,bearish,1,,,,...,-0.11,-0.11,-0.11,-0.11,-0.12,-0.12,-0.12,-0.12,-0.12,-0.12
4,1780084800,1.1672,1.16734,1.16584,1.16606,bearish,1,,,,...,-0.10,-0.10,-0.10,-0.10,0.10,0.21,-0.10,-0.10,-0.10,-0.10


In [5]:
print(df["15m_time"].nunique())
print(len(df))

1536
1536


In [6]:
target_cols = [
    "buy_10","buy_20","buy_30","buy_40","buy_50","buy_60",
    "sell_10","sell_20","sell_30","sell_40","sell_50","sell_60"
]

print(df[target_cols].isna().mean().sort_values())

buy_10     0.134766
buy_20     0.134766
buy_30     0.134766
buy_40     0.134766
buy_50     0.134766
buy_60     0.134766
sell_10    0.134766
sell_20    0.134766
sell_30    0.134766
sell_40    0.134766
sell_50    0.134766
sell_60    0.134766
dtype: float64


In [7]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1536 entries, 0 to 1535
Data columns (total 51 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   4h_time       1536 non-null   object 
 1   4h_open       1536 non-null   object 
 2   4h_high       1536 non-null   object 
 3   4h_low        1536 non-null   object 
 4   4h_close      1536 non-null   object 
 5   4h_dir        1536 non-null   object 
 6   4h_group      1536 non-null   object 
 7   4h_bias       1536 non-null   object 
 8   4h_phase      1536 non-null   object 
 9   4h_momentum   1536 non-null   object 
 10  4h_pattern    1536 non-null   object 
 11  4h_atr        1536 non-null   object 
 12  4h_rsi        1536 non-null   object 
 13  1h_time       1536 non-null   object 
 14  1h_open       1536 non-null   object 
 15  1h_high       1536 non-null   object 
 16  1h_low        1536 non-null   object 
 17  1h_close      1536 non-null   object 
 18  1h_dir        1536 non-null 